In [ ]:
import strawberryfields as sf
from strawberryfields.ops import *
import numpy as np
from scipy.special import erfc
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from helper_functions import protocols
from scipy.optimize import curve_fit

In [ ]:
from importlib import reload
reload(protocols)

## Theoretical results (Helstrom Bound) CS vs DSS 

In [61]:
N_helstrom = np.arange(0, 2, 0.02)
b_helstrom = np.arange(0, 1, 0.01)

# Theoretical surface
N_mesh, beta_mesh = np.meshgrid(N_helstrom, b_helstrom, indexing="ij")
p_err_theory = protocols.helstrom_bound(N_mesh, beta_mesh)

z_theory = p_err_theory
z_theory_cs = protocols.helstrom_bound(N_mesh, 0 * beta_mesh)

# Figure
fig = go.Figure()

# Theory surface
fig.add_trace(
    go.Surface(
        x=N_mesh,
        y=beta_mesh,
        z=z_theory,
        colorscale="Reds_r",
        opacity=1,
        showscale=False,
        name="Theory"
    )
)

fig.add_trace(
    go.Surface(
        x=N_mesh,
        y=beta_mesh,
        z=z_theory_cs,
        colorscale="Blues_r",
        opacity=1,
        showscale=False,
        name="Theory"
    )
)

# Layout
fig.update_layout(
    title="Helstrom Error probability",
    width=800,
    height=800,
    scene=dict(
        xaxis_title="N",
        yaxis_title="β",
        zaxis=dict(
            title="P_err",
            type="log"
        ),
        aspectmode="cube"
    ),
)

fig.show()

# Coherent States Noise-Free

## Produce Data

In [ ]:
num_samples=3000
alpha_grid = np.linspace(0, np.sqrt(2), 100)
N_cs, p_err_cs = protocols.perr_cs(alpha_grid=alpha_grid, homodyne_angle=0, num_samples=num_samples)

np.savez(f"data/perr_data_cs_a{len(alpha_grid)}_S{num_samples}.npz", N_cs=N_cs, alpha_grid=alpha_grid, p_err_cs=p_err_cs)

## Load Data

In [ ]:
data = np.load("data/perr_data_cs_a100_S1000.npz")

N_cs = data["N_cs"]
alpha_grid = data["alpha_grid"]
p_err_cs = data["p_err_cs"]
beta_cs = 1 - N_cs/np.max(N_cs)

## Plot Data + Theory

In [ ]:

# Theoretical surface
N_mesh, beta_mesh = np.meshgrid(N_cs, beta_cs, indexing="ij")
p_err_theory = protocols.perr_homodyne(N_mesh, beta=0)

z_theory = p_err_theory
z_sim = p_err_cs

# Figure
fig = go.Figure()

# Theory surface
fig.add_trace(
    go.Surface(x=N_mesh, y=beta_mesh, z=z_theory, colorscale="Greys_r", opacity=0.5, name="Theory", colorbar=dict(title="log₁₀(P_err)"))
)

# Simulation scatter extended in beta
beta_values = np.linspace(0, 1, 20)

for beta in beta_values:
    fig.add_trace(
        go.Scatter3d(x=N_cs, y=np.full_like(N_cs, beta), z=z_sim, mode="markers",
            marker=dict(
                size=3,
                color=z_sim,
                colorscale="Viridis",
                opacity=1
            ),
            name=f"Simulation β={beta:.2f}",
            showlegend=False
        )
    )

# Layout
fig.update_layout(
    title="Error probability",
    width=800,
    height=800,
    scene=dict(
        xaxis_title="N",
        yaxis_title="β",
        zaxis = dict(title="P_err", type="log")
    )
)

fig.show()

## Plot fit of data to theory

In [60]:
# Fit theoretical form
z_data = p_err_cs

def model(N, A, B):
    return A * erfc(B*np.sqrt(N))

params, covariance = curve_fit(model, N_cs, z_data)
A_fit, B_fit = params
A_err, B_err = np.sqrt(np.diag(covariance))

print(f"A = {A_fit:.3f} ± {A_err:.3f}, {(np.abs(A_fit - 0.5)/(A_err)):.3f}σ away from theoretical value")
print(f"B = {B_fit:.3f} ± {B_err:.3f}, {(np.abs(B_fit - np.sqrt(2))/(B_err)):.3f}σ away from theoretical value")

# Create fitted surface
N_fit = np.linspace(N_cs.min(), N_cs.max(), 200)
beta_fit = np.linspace(0, 1, 200)
N_surface, beta_surface = np.meshgrid(N_fit, beta_fit, indexing="ij")

# No beta dependence
z_surface = model(N_surface, A_fit, B_fit)


# Plot
fig = go.Figure()

# Fitted surface
fig.add_trace(
    go.Surface(x=N_surface, y=beta_surface, z=z_surface, colorscale="Blues_r", opacity=1, name="Fitted surface", showscale=False)
)

fig.update_layout(
    title="Fitted error probability surface",
    width=800,
    height=800,
    scene=dict(xaxis_title="N", yaxis_title="β",
    zaxis = dict(title="P_err", type="log")
))

fig.show()

A = 0.499 ± 0.002, 0.423σ away from theoretical value
B = 1.410 ± 0.007, 0.577σ away from theoretical value


# DSS Noise-Free

## Produce Data

In [ ]:
N_grid = np.arange(0, 2, 0.04)
beta_grid = np.arange(0, 1, 0.02)
num_samples=2000
p_err = protocols.perr_dss(N_grid=N_grid, beta_grid=beta_grid, homodyne_angle=0, num_samples=num_samples)
np.savez(f"data/perr_data_dss_N{len(N_grid)}_b{len(beta_grid)}_S{num_samples}.npz", N_grid=N_grid, beta_grid=beta_grid, p_err=p_err)

## Load data

In [54]:
data = np.load("data/perr_data_dss_N50_b50_S2000.npz")

N_grid = data["N_grid"]
beta_grid = data["beta_grid"]
p_err = data["p_err"]

## Plot Data + Theory

In [59]:
# Create Mesh
N_mesh, beta_mesh = np.meshgrid(N_grid, beta_grid, indexing='ij')
p_err_theory = protocols.perr_homodyne(N_mesh, beta_mesh)

# Log scale for the probability
z_sim = p_err
z_theory = p_err_theory

# Figure
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=N_mesh.ravel(), y=beta_mesh.ravel(), z=z_sim.ravel(), mode='markers', marker=dict(size=3, color=z_sim.ravel(), colorscale='Viridis', opacity=1), name="Simulation"))
fig.add_trace(go.Surface(x=N_mesh, y=beta_mesh, z=z_theory, colorscale="Greys_r", opacity=0.5, showscale=False))

# Figure settings
fig.update_layout(
    title="Error probability", 
    scene=dict(xaxis_title="N", 
               yaxis_title="β",  
               zaxis=dict(title="P_err", type="log")), 
                width=800, 
                height=800
                )
fig.show()

## Plot fit of data to theory

In [58]:
N_mesh, beta_mesh = np.meshgrid(N_grid, beta_grid, indexing="ij")

# Define fit function
def model(coords, A, B):

    N, beta = coords
    alpha = np.sqrt(N*(1-beta))
    Sigma = 1 / (np.sqrt(N*beta) +np.sqrt(1 + N*beta))

    return (A * erfc(B*alpha/Sigma)).ravel()

# Fit A and B
params, covariance = curve_fit(model,(N_mesh, beta_mesh), p_err.ravel())
A_fit, B_fit = params
A_err, B_err = np.sqrt(np.diag(covariance))
print(f"A = {A_fit:.3f} ± {A_err:.3f}, {(np.abs(A_fit - 0.5)/(A_err)):.3f}σ away from theoretical value")
print(f"B = {B_fit:.3f} ± {B_err:.3f}, {(np.abs(B_fit - np.sqrt(2))/(B_err)):.3f}σ away from theoretical value")


# Generate fitted surface
N_fit = np.linspace(N_mesh.min(), N_mesh.max(), 200)
beta_fit = np.linspace(beta_mesh.min(), beta_mesh.max(), 200)
N_surface, beta_surface = np.meshgrid(N_fit, beta_fit, indexing="ij")
p_surface = model((N_surface, beta_surface), A_fit, B_fit).reshape(N_surface.shape)

# Log scale
z_sim = p_err.ravel()
z_surface = p_surface

# Plot
fig = go.Figure()

# Fitted surface
fig.add_trace(
    go.Surface(x=N_surface, y=beta_surface, z=z_surface, colorscale="Greens_r", opacity=1, name="Fit", showscale=False)
)

# Layout
fig.update_layout(
    title="Error probability fit",
    width=700,
    height=700,
    scene=dict(
        xaxis_title="N",
        yaxis_title="β",
        zaxis=dict(title="P_err", type="log")
    )
)


fig.show()

A = 0.500 ± 0.001, 0.408σ away from theoretical value
B = 1.415 ± 0.002, 0.556σ away from theoretical value
